#  Clasificación de imágenes de flores usando TensorFlow

adaptado de: https://www.tensorflow.org/tutorials/images/classification

En este tutorial se muestra cómo clasificar imágenes de flores utilizando un modelo secuencial de tf.keras y cómo cargar datos con la utilidad `tf.keras.utils.image_dataset_from_directory`. Además, se cubren conceptos como la carga eficiente de un conjunto de datos desde el disco, la identificación del sobreajuste y las técnicas para mitigarlo, como la ampliación de datos (data augmentation) y el dropout.

Este tutorial sigue un flujo básico de aprendizaje automático:

1. Examinación y comprensión de los datos.
3. Construcción del pipeline de entrada.
3. Construcción del modelo.
4. Entrenamiento del modelo.
5. Prueba del modelo.
6. Mejora del modelo y repetición del proceso.
7. Convertir el modelo a TensorFlow Lite


En la última parte del tutorial se muestra cómo convertir un modelo guardado en un modelo TensorFlow Lite para su uso en dispositivos móviles, embebidos y IoT.

## Instalar las librerias

Se importan las bibliotecas necesarias para cargar y procesar imágenes, construir el modelo y realizar la visualización de resultados.



In [ ]:
# Importar bibliotecas necesarias
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential


## Descargar y explorar el conjunto de datos

Este taller utiliza un conjunto de datos de aproximadamente 3,700 fotos de flores, que se encuentran en 5 subdirectorios, uno para cada clase: margaritas, diente de león, rosas, girasoles y tulipanes.

La función `tf.keras.utils.get_file` es una utilidad que proporciona TensorFlow para descargar archivos desde una URL y almacenarlos de manera local en el sistema de archivos.

Este archivo descargado en Archivo descargado en: /root/.keras/datasets/dataset.tar puede ser utilizado posteriormente en tu flujo de trabajo de TensorFlow, como en el caso de modelos preentrenados o datasets.

In [ ]:
import tensorflow as tf
import os

# URL del archivo .tar
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

# Especifica el directorio de trabajo donde quieres guardar el archivo descomprimido
working_dir = './'

# Asegúrate de que el directorio de trabajo exista
os.makedirs(working_dir, exist_ok=True)

# Descargar y extraer el archivo en el directorio de trabajo
data_dir = tf.keras.utils.get_file(
    'flower_photos.tgz',  # Nombre con el que se descargará el archivo
    origin=dataset_url,  # URL del archivo .tgz
    untar=True,           # Descomprimir el archivo
    cache_dir=working_dir # Directorio donde se almacenará
)

# Imprimir la ruta del directorio donde se extrajeron los archivos
print(f"Archivo descargado y extraído en: {data_dir}")


##Organizar tus imágenes para crear tu propio conjunto de datos

Cuando quieres usar tus propias imágenes en TensorFlow para clasificación, es fundamental organizarlas correctamente. La estructura recomendada es colocar cada tipo de imagen en subcarpetas, donde el nombre de cada subcarpeta representa la clase de las imágenes. Aquí tienes un ejemplo de cómo estructurar tu conjunto de datos:

```so
flower_photos/               <-- Raíz del directorio
    daisy/                    <-- Carpeta para imágenes de "daisy"
        daisy1.jpg
        daisy2.jpg
        ...
    dandelion/                <-- Carpeta para imágenes de "dandelion"
        dandelion1.jpg
        dandelion2.jpg
        ...
    roses/                    <-- Carpeta para imágenes de "roses"
        rose1.jpg
        rose2.jpg
        ...
    sunflowers/               <-- Carpeta para imágenes de "sunflowers"
        sunflower1.jpg
        sunflower2.jpg
        ...
    tulips/                   <-- Carpeta para imágenes de "tulips"
        tulip1.jpg
        tulip2.jpg
        ...
```

Si las imagenes las tiene el directorio no es necesario hacer el get. Pero si la tiene en una pagina web comprimida debe descargar el dataset de imágenes y luego cargarlo usando tf.keras.utils.get_file y tf.keras.utils.image_dataset_from_directory.

Reafirmamos que debes organizar las imágenes de una manera específica y asegurarte de que estén preparadas correctamente para su carga y uso con TensorFlow.

##Contar el número de imágenes en un directorio específico

Este código se utiliza para contar el número total de imágenes con la extensión .jpg dentro de un directorio y sus subdirectorios. Utiliza la librería pathlib de Python para navegar por el sistema de archivos y contar las imágenes de manera eficiente.

In [ ]:
from pathlib import Path
# Definir el directorio como un objeto Path
dir = Path('/content/datasets/flower_photos')

# Contar las imágenes con la extensión .jpg en los subdirectorios
image_count = len(list(dir.glob('*/*.jpg')))  # Cuenta las imágenes .jpg en subdirectorios

# Imprimir el número de imágenes
print(dir)
print(image_count)

#Abrir la primera imagen de un directorio específico utilizando PIL y pathlib

Este código permite abrir la primera imagen en una carpeta llamada roses dentro de un directorio específico utilizando la librería PIL (Pillow) para manejar imágenes y pathlib para navegar por el sistema de archivos.

Se asume que las imágenes están almacenadas en un directorio de trabajo y se busca la primera imagen dentro de la carpeta roses para cargarla y mostrarla.

In [ ]:
roses = list(dir.glob('roses/*'))
PIL.Image.open(str(roses[0]))

In [ ]:
#Abrir la segunda imagen de un directorio específico utilizando PIL y pathlib
PIL.Image.open(str(roses[1]))

#Abrir la primera imagen de un directorio "tulips" utilizando PIL y pathlib

Este código permite abrir la primera imagen dentro de una subcarpeta llamada tulips en un directorio específico utilizando la librería PIL (Pillow) para manejar imágenes y pathlib para navegar por el sistema de archivos. El código obtiene la primera imagen de la carpeta tulips y la muestra. Asegúrate de que haya al menos una imagen en la carpeta tulips para evitar errores.

In [ ]:
tulips = list(dir.glob('tulips/*'))
PIL.Image.open(str(tulips[0]))

In [ ]:
PIL.Image.open(str(tulips[1]))

## Cargar un dataset de imágenes y dividirlo en entrenamiento y validación utilizando image_dataset_from_directory de TensorFlow


Usamos tf.keras.utils.image_dataset_from_directory para cargar las imágenes desde el directorio. Además, especificamos parámetros como el tamaño de las imágenes y el tamaño del lote para el entrenamiento.

Además, divide automáticamente el dataset en un conjunto de entrenamiento y otro de validación utilizando un split de 80-20, respectivamente. Las imágenes son redimensionadas a un tamaño especificado (img_height y img_width), y los datos se cargan en lotes (batch_size). La división en entrenamiento y validación se realiza de manera aleatoria, y se fija una semilla (seed) para la reproducibilidad.



El **batch_size** es uno de los hiperparámetros más importantes en el entrenamiento de modelos de aprendizaje automático, especialmente en redes neuronales profundas.

Definir un batch_size adecuado tiene un impacto directo en la eficiencia del entrenamiento, el rendimiento del modelo y los recursos computacionales requeridos.

No existe un valor único que sea óptimo para todos los problemas, pero se pueden seguir algunas pautas generales y estrategias para encontrar un valor que funcione bien para tu caso específico.

**Factores que afectan el batch_size óptimo:**

* Memoria de la GPU/CPU:

  El batch_size define cuántas muestras se procesan a la vez antes de actualizar los pesos del modelo. A mayor tamaño de lote, mayor será la memoria necesaria, ya que más datos deben almacenarse en la memoria para el cálculo del gradiente.

* Recursos limitados:

  Si estás entrenando en una GPU con memoria limitada, puedes necesitar reducir el batch_size para evitar errores de memoria. La cantidad máxima de memoria que puedes usar depende de tu hardware.

* Uso eficiente de la GPU:

Para obtener un uso óptimo de la GPU, a menudo es útil elegir un batch_size lo suficientemente grande como para aprovechar completamente la capacidad de la GPU, pero sin que se produzcan errores de memoria.

* Velocidad de convergencia:

  Un batch_size pequeño (por ejemplo, 16 o 32) tiende a hacer el proceso de optimización más ruidoso, lo que puede ayudar al modelo a escapar de los óptimos locales y explorar mejor el espacio de búsqueda. Sin embargo, esto puede resultar en una convergencia más lenta y un tiempo de entrenamiento más largo.

  Un batch_size grande (por ejemplo, 256 o 512) proporciona gradientes más estables y menos ruido, lo que puede hacer que el modelo converja más rápidamente, pero puede estar atrapado en óptimos locales o plateaus.

* Tiempo de entrenamiento:

  Batch sizes grandes generalmente reducen el número de actualizaciones de parámetros por época, lo que acelera el entrenamiento (menos pasos por época), pero el tiempo para cada paso de entrenamiento puede ser más largo debido al mayor uso de memoria.

  Batch sizes pequeños generan más actualizaciones de parámetros por época, lo que puede hacer que el modelo se actualice más frecuentemente y tal vez se beneficie de una mayor exploración, pero también pueden ralentizar el proceso debido a la sobrecarga computacional.

* Regularización:

  Un batch_size pequeño tiene un efecto regularizador debido al "ruido" en los gradientes. El ruido hace que el modelo explore diferentes soluciones en el espacio de parámetros, lo que puede mejorar la generalización.

  Un batch_size grande puede tener un efecto de regularización más débil, ya que los gradientes son más estables y el modelo puede sobreajustarse si no se ajusta adecuadamente.

In [ ]:
batch_size = 32
img_height = 180
img_width = 180


train_ds = tf.keras.utils.image_dataset_from_directory(
  dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)


El dataset tiene 3670 archivos de imágenes en total, distribuidos en 5 clases diferentes. Al usar un validation_split=0.2, el dataset se divide en:

80% de los archivos (2936 imágenes) se destinan al conjunto de entrenamiento.
20% de los archivos (734 imágenes) se destinan al conjunto de validación.
Este enfoque es útil para entrenar y evaluar un modelo de aprendizaje automático, ya que el conjunto de validación proporciona una manera de medir el rendimiento del modelo en datos que no ha visto durante el entrenamiento.

# Cargar las clases de un dataset y traducirlas a español utilizando un diccionario

Este código carga las clases del dataset de imágenes utilizando train_ds.class_names, que devuelve una lista con los nombres de las clases en inglés (o en el idioma en el que fueron etiquetadas las carpetas del dataset).

`Image_dataset_from_directory` carga las imágenes de esas subcarpetas y las etiqueta automáticamente según el nombre de la subcarpeta.

Luego, utilizando un diccionario, traduce estos nombres de clase a español para mayor comprensión. Finalmente, imprime las clases traducidas.

In [ ]:
# Cargar las clases del dataset de entrenamiento
class_names = train_ds.class_names

# Diccionario que mapea los nombres de las clases en inglés a español
class_translation = {
    'daisy': 'Margaritas',
    'dandelion': 'Diente de Leon',
    'roses': 'Rosas',
    'sunflowers': 'Girasoles',
    'tulips': 'Tulipanes'
    }

# Traducir las clases al español utilizando el diccionario
translated_class_names = [class_translation.get(class_name, class_name) for class_name in class_names]

# Imprimir las clases traducidas
print(translated_class_names)


#Visualizar un lote de imágenes del dataset de entrenamiento con sus etiquetas

Este código utiliza matplotlib para mostrar una cuadrícula de imágenes de un lote de entrenamiento extraído de train_ds. Se visualizan las primeras 9 imágenes junto con sus respectivas etiquetas.

La función `train_ds.take(1)` obtiene un solo lote del dataset de entrenamiento, y luego se muestra cada imagen con su correspondiente título, que es el nombre de la clase (según el índice de la etiqueta de la imagen).

In [ ]:
import matplotlib.pyplot as plt

# Configurar el tamaño de la figura para la visualización de las imágenes
plt.figure(figsize=(10, 10))

# Tomar un lote de imágenes y etiquetas del dataset de entrenamiento
for images, labels in train_ds.take(1):
    # Iterar a través de las primeras 9 imágenes del lote
    for i in range(9):
        # Crear una subgráfica para cada imagen en una cuadrícula de 3x3
        ax = plt.subplot(3, 3, i + 1)

        # Mostrar la imagen actual en el gráfico
        plt.imshow(images[i].numpy().astype("uint8"))  # Convertir la imagen a formato uint8 para mostrarla correctamente
        # Asignar el título de la subgráfica a la clase correspondiente de la etiqueta
        plt.title(translated_class_names[labels[i]])  # `labels[i]` es el índice de la clase de la imagen
        plt.axis("off")  # Ocultar los ejes para que solo se vea la imagen

# Mostrar la visualización completa
plt.show()


#Verificar las formas de las imágenes y etiquetas en un lote del dataset

Este código imprime las dimensiones de un lote de imágenes y sus etiquetas en el dataset de entrenamiento train_ds.

La estructura del lote es útil para verificar cómo están organizados los datos y asegurarse de que tienen la forma esperada antes de entrenar el modelo.


In [ ]:
# Iterar sobre el dataset de entrenamiento
for image_batch, labels_batch in train_ds:
    # Imprimir la forma (dimensiones) del lote de imágenes
    print(image_batch.shape)

    # Imprimir la forma (dimensiones) del lote de etiquetas
    print(labels_batch.shape)

    # Salir del bucle después de imprimir la información del primer lote
    break


Esto indica que el primer lote contiene 32 imágenes con un tamaño de 180x180 píxeles y 3 canales (RGB), y las etiquetas del lote tienen una forma (32,), lo que significa que hay 32 etiquetas, cada una correspondiente a una de las 32 imágenes en el lote.

##Optimización del pipeline de datos con cache, shuffle, y prefetch en TensorFlow

Este código mejora el rendimiento del pipeline de datos en TensorFlow al aplicar técnicas como caché, mezcla y prefetching.

Estas optimizaciones permiten que el modelo de entrenamiento aproveche mejor los recursos del sistema (como la CPU/GPU) y acelere el proceso de entrenamiento, especialmente cuando el acceso a los datos es un cuello de botella.




In [ ]:
AUTOTUNE = tf.data.AUTOTUNE  # Habilitar ajuste automático para prefetch

# Aplicar optimizaciones al conjunto de entrenamiento
train_ds = train_ds.cache()\
                   .shuffle(1000)\
                   .prefetch(buffer_size=AUTOTUNE)

# Aplicar optimizaciones al conjunto de validación
val_ds = val_ds.cache()\
               .prefetch(buffer_size=AUTOTUNE)



**train_ds.cache():**

Guarda en memoria el conjunto de entrenamiento después de la primera carga, acelerando el acceso a los datos en las épocas siguientes.

**train_ds.shuffle(1000):**

Mezcla aleatoriamente los datos del conjunto de entrenamiento para evitar el aprendizaje de patrones espurios y mejorar la generalización.


**train_ds.prefetch(buffer_size=AUTOTUNE):**

Permite que los datos se carguen en segundo plano mientras el modelo está entrenando, minimizando el tiempo de inactividad de la GPU o CPU.


**val_ds.cache():**

Similar al conjunto de entrenamiento, pero solo se utiliza cache() para el conjunto de validación, ya que no se necesita mezclar los datos en la validación.


**val_ds.prefetch(buffer_size=AUTOTUNE):**

Optimiza la carga de los datos de validación mientras el modelo está entrenando, asegurando que los datos estén listos para la siguiente época.

**Beneficios:**

1. Mejora en el rendimiento: Al utilizar cache(), los datos no tienen que leerse desde el disco después de la primera época, lo que acelera el entrenamiento.

2. Reducción de tiempos de espera: shuffle() y prefetch() permiten que el modelo continúe trabajando en la próxima iteración mientras los datos se preparan en segundo plano, reduciendo el tiempo de inactividad.

3. Aprovechamiento de recursos: Usar AUTOTUNE para prefetch() asegura que el proceso de entrada de datos se ajuste dinámicamente a los recursos disponibles, maximizando el rendimiento en función de la capacidad del sistema.


## Normalización de imágenes con Rescaling en TensorFlow

La capa Rescaling en TensorFlow se utiliza para normalizar las imágenes de entrada, es decir, para cambiar el rango de los valores de píxeles de las imágenes. Por lo general, se utiliza para ajustar los valores de píxeles de imágenes RGB de un rango de [0, 255] a un rango de [0, 1], lo que facilita el proceso de aprendizaje y mejora la estabilidad y el rendimiento del modelo.

En este caso, se utiliza para escalar los valores de los píxeles dividiendo por 255, transformando los valores de las imágenes de [0, 255] a [0, 1].

In [ ]:
normalization_layer = layers.Rescaling(1./255)


##Verificación de la normalización de imágenes con Rescaling en TensorFlow

Este código utiliza la capa de normalización Rescaling para escalar los valores de los píxeles de las imágenes del conjunto de datos de entrenamiento de un rango de [0, 255] a [0, 1].

Después, se verifica que la normalización se haya aplicado correctamente mostrando los valores mínimo y máximo de los píxeles de la primera imagen del lote, que deberían estar dentro del rango [0, 1].



In [ ]:
# Aplicar la normalización a las imágenes utilizando map()
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))

# Extraer un lote de imágenes y etiquetas del conjunto de datos normalizado
image_batch, labels_batch = next(iter(normalized_ds))

# Tomar la primera imagen del lote
first_image = image_batch[0]

# Verificar que los valores de los píxeles estén en el rango [0, 1]
print(np.min(first_image), np.max(first_image))



0.0: Este valor representa el valor mínimo de un píxel en la imagen. Después de normalizar los valores de píxel al rango de [0, 1] (dividiendo por 255), algunos píxeles pueden tener el valor más bajo de 0. Esto podría ocurrir, por ejemplo, si un píxel en la imagen original tenía el valor más bajo de 0 (negro o un color muy oscuro).


0.99638706: Este es el valor máximo de un píxel en la imagen después de la normalización. En este caso, el valor más alto es 0.99638706, lo cual es ligeramente inferior a 1.0. Esto puede ocurrir si el valor máximo original de un píxel en la imagen no era exactamente 255 (el valor máximo en la escala original de [0, 255]). En algunas imágenes, los valores de los píxeles pueden estar distribuidos de manera que no alcanzan el valor máximo posible (255), lo que resulta en un valor máximo ligeramente inferior después de la normalización.



## Definición de un modelo CNN para clasificación de imágenes

Este código define una red neuronal convolucional (CNN) utilizando la API Sequential de TensorFlow/Keras para la clasificación de imágenes. La red incluye varias capas convolucionales y de agrupamiento (max pooling), seguidas de una capa densa para la clasificación final. La normalización de las imágenes se realiza antes de pasar las imágenes a las capas convolucionales para mejorar la eficiencia del entrenamiento.

In [ ]:
from tensorflow.keras import layers, Input
# Número de clases (etiquetas) para clasificación
num_classes = len(class_names)

# Definición del modelo CNN
model = Sequential([
  # Normalización de las imágenes, escalando los valores de los píxeles a [0, 1]
  Input(shape=(img_height, img_width, 3)),  # Especificar la forma de entrada con la capa Input

  layers.Rescaling(1./255),


  # Primera capa convolucional: 16 filtros, tamaño del kernel 3x3, padding 'same', activación ReLU
  layers.Conv2D(16, 3, padding='same', activation='relu'),

  # Capa de MaxPooling para reducir las dimensiones de la imagen
  layers.MaxPooling2D(),

  # Segunda capa convolucional: 32 filtros, tamaño del kernel 3x3, padding 'same', activación ReLU
  layers.Conv2D(32, 3, padding='same', activation='relu'),

  # Capa de MaxPooling para reducir las dimensiones de la imagen
  layers.MaxPooling2D(),

  # Tercera capa convolucional: 64 filtros, tamaño del kernel 3x3, padding 'same', activación ReLU
  layers.Conv2D(64, 3, padding='same', activation='relu'),

  # Capa de MaxPooling para reducir las dimensiones de la imagen
  layers.MaxPooling2D(),

  # Aplanar la salida 2D para convertirla en un vector 1D
  layers.Flatten(),

  # Capa densa con 128 neuronas y activación ReLU
  layers.Dense(128, activation='relu'),

  # Capa de salida con un número de neuronas igual al número de clases
  layers.Dense(num_classes)
])


In [ ]:
model.summary()

##Compilación del modelo con optimizador, función de pérdida y métricas en TensorFlow

Este código configura la compilación de un modelo en Keras (parte de TensorFlow) utilizando el optimizador Adam, la función de pérdida SparseCategoricalCrossentropy (con logits) y la métrica de precisión (accuracy).

La compilación de un modelo es un paso esencial antes de entrenarlo, ya que define cómo el modelo aprenderá y evaluará su desempeño durante el entrenamiento.

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])


##Entrenar el modelo

Entrenamiento del modelo en TensorFlow con un número determinado de épocas

Este código entrena el modelo previamente definido utilizando el conjunto de datos de entrenamiento (train_ds) y el conjunto de datos de validación (val_ds). El entrenamiento se realiza durante un número de épocas especificado, en este caso, 10 épocas.

La función model.fit() ajusta los parámetros del modelo (pesos y sesgos) para minimizar la función de pérdida y mejorar las predicciones del modelo. Durante cada época, se evalúa el rendimiento tanto en el conjunto de entrenamiento como en el de validación.

In [ ]:
epochs=10
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

## Visualización de la precisión y la pérdida durante el entrenamiento y la validación

Este código visualiza el rendimiento del modelo a lo largo de las épocas de entrenamiento, mostrando tanto la precisión como la pérdida para los datos de entrenamiento y validación.

Las gráficas resultantes permiten observar cómo el modelo está aprendiendo, así como si está sobreajustándose (overfitting) o generalizando correctamente.

In [ ]:
# Obtener los valores de precisión y pérdida para el entrenamiento y la validación desde el historial de entrenamiento.
acc = history.history['accuracy']  # Precisión del modelo en el conjunto de entrenamiento.
val_acc = history.history['val_accuracy']  # Precisión del modelo en el conjunto de validación.

loss = history.history['loss']  # Pérdida del modelo en el conjunto de entrenamiento.
val_loss = history.history['val_loss']  # Pérdida del modelo en el conjunto de validación.

# Definir el rango de épocas (en este caso, de 0 a 'epochs' - 1)
epochs_range = range(epochs)

# Crear una figura de tamaño 8x8 para contener las dos gráficas.
plt.figure(figsize=(8, 8))

# Subgráfico 1: Gráfica de precisión (accuracy).
plt.subplot(1, 2, 1)  # (1, 2, 1) significa 1 fila, 2 columnas, y esta es la primera subgráfica.
plt.plot(epochs_range, acc, label='Training Accuracy')  # Graficar la precisión del entrenamiento.
plt.plot(epochs_range, val_acc, label='Validation Accuracy')  # Graficar la precisión de la validación.
plt.legend(loc='lower right')  # Mostrar la leyenda en la esquina inferior derecha.
plt.title('Training and Validation Accuracy')  # Título de la gráfica.

# Subgráfico 2: Gráfica de pérdida (loss).
plt.subplot(1, 2, 2)  # (1, 2, 2) significa 1 fila, 2 columnas, y esta es la segunda subgráfica.
plt.plot(epochs_range, loss, label='Training Loss')  # Graficar la pérdida del entrenamiento.
plt.plot(epochs_range, val_loss, label='Validation Loss')  # Graficar la pérdida de la validación.
plt.legend(loc='upper right')  # Mostrar la leyenda en la esquina superior derecha.
plt.title('Training and Validation Loss')  # Título de la gráfica.

# Mostrar la figura con las dos subgráficas.
plt.show()


Las gráficas muestran que la precisión de entrenamiento y la precisión de validación difieren por márgenes amplios, y el modelo ha logrado solo alrededor del 60% de precisión en el conjunto de validación.

Las siguientes secciones del taller se muestra cómo ajustan parámetros para aumentar el rendimiento general del modelo.

## Overfitting

En las gráficas anteriores, la precisión de entrenamiento aumenta de manera lineal a lo largo del tiempo, mientras que la precisión de validación se estabiliza alrededor del 60% durante el proceso de entrenamiento. Además, es evidente la diferencia entre la precisión de entrenamiento y la de validación, lo que indica un posible sobreajuste.

Cuando se dispone de un número reducido de ejemplos de entrenamiento, el modelo a veces aprende patrones erróneos o detalles no relevantes de los datos, lo que afecta negativamente su desempeño al enfrentarse a nuevos ejemplos. Este fenómeno se conoce como sobreajuste, lo que significa que el modelo tendrá dificultades para generalizar en un nuevo conjunto de datos.

Existen varias estrategias para combatir el sobreajuste durante el proceso de entrenamiento. En este tutorial, se utilizarán aumento de datos y se añadirá dropout al modelo.

## Data Augmentation y las utilidades de Keras

Data Augmentation (aumento de datos) es una técnica utilizada para mejorar la generalización de un modelo, especialmente cuando se tiene un conjunto de datos limitado. La idea es generar nuevas variaciones de los datos de entrenamiento existentes, aplicando transformaciones aleatorias a las imágenes, lo que permite que el modelo vea más ejemplos y aprenda a generalizar mejor. En lugar de entrenar con un número fijo de imágenes, el modelo se entrena con versiones modificadas de esas imágenes, lo que reduce el riesgo de sobreajuste.

Este proceso es muy útil cuando el modelo está sobreajustando (overfitting), ya que ayuda a simular diferentes condiciones y escenarios para que el modelo no dependa demasiado de características específicas de las imágenes de entrenamiento originales.



## Utilización de tf.keras.layers.RandomFlip, tf.keras.layers.RandomRotation y tf.keras.layers.RandomZoom

TensorFlow y Keras ofrecen varias capas que pueden aplicarse a las imágenes de manera aleatoria durante el entrenamiento para realizar aumentos de datos. Las siguientes capas son algunas de las más comunes:

1. `tf.keras.layers.RandomFlip:`

**Descripción:** Esta capa aplica un flip aleatorio (reflejo) horizontal o vertical a las imágenes.

**Uso:** Se utiliza para simular variaciones en la orientación de los objetos dentro de las imágenes, lo que ayuda a mejorar la capacidad del modelo para aprender invariantes de orientación.

```python

tf.keras.layers.RandomFlip('horizontal')
```
Esto voltea las imágenes aleatoriamente en el eje horizontal.

También puede usarse en el eje vertical con **'vertical'** o en ambos ejes con **'horizontal_and_vertical'**.


2. `tf.keras.layers.RandomRotation:`

**Descripción:** Esta capa aplica una rotación aleatoria a las imágenes.

**Uso:** La rotación permite al modelo aprender a reconocer los objetos independientemente de su orientación en la imagen.

Esto es útil cuando el modelo puede encontrarse con objetos rotados en datos no vistos.


```python

tf.keras.layers.RandomRotation(0.2)
```
Este ejemplo rota las imágenes aleatoriamente en un rango de -20% a +20% de 360 grados.

El parámetro 0.2 indica el rango de rotación, donde 1 sería una rotación de 360 grados (una rotación completa).

3.  `tf.keras.layers.RandomZoom:`

**Descripción:** Esta capa aplica un zoom aleatorio a las imágenes.

**Uso:** El zoom simula situaciones en las que el objeto de interés puede estar más cerca o más lejos en la imagen.
Esto ayuda al modelo a ser más robusto a los cambios de escala de los objetos.

```python
tf.keras.layers.RandomZoom(0.2)
```

En este ejemplo, el zoom puede hacer que las imágenes se acerquen o se alejen aleatoriamente en un rango de hasta el 20%. Un valor de 0.2 indica que se puede aplicar un zoom del 20% hacia adentro o hacia afuera.

Vamos a aplicar estas técnicas.

##Implementación de Aumento de Datos (Data Augmentation) en Keras

En este ejemplo, se crea un pipeline de aumento de datos que aplica tres transformaciones aleatorias a las imágenes: volteo horizontal, rotación y zoom.

Estas transformaciones aumentan la diversidad de las imágenes de entrenamiento, lo que permite que el modelo aprenda de una mayor variedad de escenarios y mejore su rendimiento en datos no vistos.

In [ ]:
data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal",
                      input_shape=(img_height,
                                  img_width,
                                  3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

##Visualización de Imágenes Aumentadas con Data Augmentation en Keras

Este código muestra cómo visualizar imágenes aumentadas (con transformaciones aleatorias como rotación, zoom, y flip horizontal) durante el proceso de entrenamiento. Se utiliza un pipeline de aumento de datos previamente definido y se aplica a un lote de imágenes de entrenamiento (train_ds). Posteriormente, se visualizan las imágenes aumentadas en una cuadrícula de 3x3 para mostrar la variabilidad que se introduce mediante las transformaciones aleatorias.

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
  for i in range(9):
    augmented_images = data_augmentation(images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_images[0].numpy().astype("uint8"))
    plt.axis("off")

## Dropout

Uso de Dropout para Reducir el Sobreajuste en Redes Neuronales con Keras


El dropout es una técnica de regularización utilizada para reducir el sobreajuste en las redes neuronales. Consiste en "apagar" aleatoriamente una fracción de las unidades de activación durante el entrenamiento. Esto obliga al modelo a aprender representaciones más robustas y menos dependientes de cualquier neurona en particular. En este caso, se va a integrar la capa Dropout en una red neuronal antes de entrenarla usando imágenes aumentadas (data augmentation).

Este modelo es una red neuronal convolucional (CNN) diseñada para clasificación multiclase. Combina varias técnicas para mejorar el rendimiento, como data augmentation (aumento de datos), dropout (regularización) y normalización. Estas técnicas ayudan a mejorar la capacidad del modelo para generalizar y prevenir el sobreajuste. El modelo está estructurado con capas convolucionales para la extracción de características, capas de pooling para reducir la dimensionalidad y capas densas para la clasificación final.

El parámetro padding='same' en las capas convolucionales de Keras (como Conv2D) determina cómo se maneja el tamaño de la salida en relación con el tamaño de la entrada al aplicar los filtros de convolución.

* **Añade ceros alrededor de la imagen de entrada (padding):**

Si la operación de convolución no puede ajustarse completamente al borde de la imagen (debido a la dimensión del filtro), se añaden ceros a los bordes de la imagen.


* **Salida con el mismo tamaño que la entrada (en dimensiones espaciales):**

El resultado de la convolución tendrá la misma altura y anchura que la entrada, salvo que se utilicen otros parámetros como un stride mayor a 1.

In [ ]:
model = Sequential([
  data_augmentation,
  layers.Rescaling(1./255),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes, name="outputs")
])

## Compilar y entrenar el modelo

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
epochs = 15
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

## Visualización de Resultados de Entrenamiento con Data Augmentation y Dropout

Después de aplicar técnicas como Data Augmentation y Dropout, podemos observar una mejora significativa en la capacidad de generalización del modelo. Esto se refleja en una menor diferencia entre la precisión de entrenamiento y la precisión de validación, lo que indica que el modelo no está sobreajustando tanto los datos de entrenamiento. A continuación, se presenta un ejemplo de cómo visualizar estos resultados para comparar la precisión y la pérdida tanto en los conjuntos de entrenamiento como de validación durante el proceso de entrenamiento.

In [ ]:

# Extraer las métricas de precisión y pérdida del historial del modelo
acc = history.history['accuracy']  # Precisión en entrenamiento
val_acc = history.history['val_accuracy']  # Precisión en validación

loss = history.history['loss']  # Pérdida en entrenamiento
val_loss = history.history['val_loss']  # Pérdida en validación

# Número de épocas
epochs_range = range(epochs)

# Crear las gráficas para visualizar los resultados de entrenamiento y validación
plt.figure(figsize=(10, 10))

# Subgráfico de Precisión
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Precisión en Entrenamiento')
plt.plot(epochs_range, val_acc, label='Precisión en Validación')
plt.legend(loc='lower right')
plt.title('Precisión de Entrenamiento y Validación')

# Subgráfico de Pérdida
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Pérdida en Entrenamiento')
plt.plot(epochs_range, val_loss, label='Pérdida en Validación')
plt.legend(loc='upper right')
plt.title('Pérdida de Entrenamiento y Validación')

# Mostrar las gráficas
plt.show()


## Predecir una nueva imagen

Este código realiza una predicción sobre una imagen específica (en este caso, una imagen de un girasol) utilizando un modelo previamente entrenado.

La imagen se descarga desde una URL, se procesa para ajustarse al tamaño adecuado, y se utiliza para realizar una predicción con el modelo. Posteriormente, se imprime la clase predicha junto con la probabilidad de confianza de la predicción.

In [ ]:

# URL de la imagen de girasol
sunflower_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"

# Descargar la imagen y guardarla en la ruta especificada
sunflower_path = tf.keras.utils.get_file('Red_sunflower', origin=sunflower_url)

# Cargar la imagen y redimensionarla al tamaño que espera el modelo
img = tf.keras.utils.load_img(
    sunflower_path, target_size=(img_height, img_width)
)

# Convertir la imagen a un array de Numpy
img_array = tf.keras.utils.img_to_array(img)

# Expandir las dimensiones para crear un lote (batch) de imágenes
img_array = tf.expand_dims(img_array, 0)  # Crea un batch con una imagen

# Realizar la predicción con el modelo
predictions = model.predict(img_array)

# Aplicar la función softmax para obtener las probabilidades de las clases
score = tf.nn.softmax(predictions[0])

# Imprimir el resultado de la predicción
print(
    "Esta imagen probablemente pertenece a {} con una probabilidad del {:.2f}%."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)


#Uso de TensorFlow Lite para Inferencia en Dispositivos Móviles y Embebidos

TensorFlow Lite es una biblioteca que permite ejecutar modelos de machine learning de manera eficiente en dispositivos móviles, embebidos y de borde (edge devices).

En lugar de utilizar TensorFlow estándar, que es más adecuado para servidores y entornos con mayor capacidad computacional, TensorFlow Lite optimiza los modelos para que puedan funcionar en dispositivos con recursos limitados.

Esta herramienta es esencial para implementar modelos de machine learning en aplicaciones móviles, IoT y otros dispositivos que requieren inferencia local.

##Conversión del Modelo Keras a TensorFlow Lite para Inferencia en Dispositivos Móviles

Para implementar modelos entrenados en dispositivos móviles o sistemas embebidos, es necesario convertir los modelos Keras (que son generalmente grandes y pesados) a un formato más pequeño y eficiente, como TensorFlow Lite.

Este formato optimizado permite que los modelos se ejecuten en dispositivos con recursos limitados, como teléfonos móviles, dispositivos IoT y otros dispositivos de borde.

En este ejemplo, se explica cómo tomar un modelo entrenado en Keras (en este caso, un modelo Sequential) y convertirlo a un modelo TensorFlow Lite utilizando tf.lite.TFLiteConverter.from_keras_model.

Este proceso genera un modelo más pequeño que puede ser usado eficientemente en dispositivos móviles.

In [ ]:

# Suponiendo que el modelo Keras ya ha sido entrenado (model)
# Convertir el modelo Keras a un modelo TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)  # Usar el modelo Keras entrenado
tflite_model = converter.convert()  # Convertir el modelo

# Guardar el modelo convertido en un archivo .tflite
tflite_model_path = 'model.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

print(f"Modelo convertido y guardado en {tflite_model_path}")


The TensorFlow Lite model you saved in the previous step can contain several function signatures. The Keras model converter API uses the default signature automatically. Learn more about [TensorFlow Lite signatures](https://www.tensorflow.org/lite/guide/signatures).

### Correr un modelo de TtensorFlow Lite

Conversión de un Modelo Keras a TensorFlow Lite con Firmas de Funciones

Cuando se convierte un modelo entrenado en Keras a TensorFlow Lite, el modelo resultante puede incluir varias firmas de funciones que describen cómo se debe interactuar con el modelo, especialmente cuando se realiza la inferencia en dispositivos móviles o embebidos.

La API del convertidor de Keras a TensorFlow Lite utiliza de manera predeterminada la firma estándar. Sin embargo, es importante entender cómo funcionan las firmas en TensorFlow Lite para escenarios más complejos donde se necesiten entradas y salidas personalizadas.

Después de convertir un modelo Keras a TensorFlow Lite, es posible realizar inferencias directamente en dispositivos con recursos limitados (como teléfonos móviles y dispositivos embebidos). TensorFlow Lite proporciona la clase tf.lite.Interpreter para cargar y ejecutar modelos en estos dispositivos. Además, es posible acceder a las firmas del modelo convertido, realizar predicciones y comparar los resultados obtenidos con los de un modelo Keras original.

En este ejemplo, se carga el modelo convertido a TensorFlow Lite, se realiza una inferencia en una imagen de muestra y se compara la predicción generada con la del modelo original para asegurarse de que ambos resultados sean casi idénticos.

##Cargar el Modelo TensorFlow Lite:
```
interpreter = tf.lite.Interpreter
```

`(model_path=TF_MODEL_FILE_PATH)`: Esta línea carga el modelo model.tflite utilizando la clase Interpreter de TensorFlow Lite.


`TF_MODEL_FILE_PATH:` Especifica la ruta al archivo del modelo TensorFlow Lite guardado anteriormente.


**Obtener la Lista de Firmas del Modelo:**

`signature_list = interpreter.get_signature_list():`

Recupera las firmas del modelo convertido. Las firmas representan las entradas y salidas del modelo, y se pueden utilizar para acceder a las funciones de predicción definidas en el modelo.

Imprime las firmas del modelo, por ejemplo,
"serving_default".

**Realizar Inferencia con la Firma Predeterminada:**

`classify_lite = interpreter.get_signature_runner`

`('serving_default'):` Obtiene la función asociada con la firma 'serving_default'. Esta es la función que se usará para realizar la inferencia.

`predictions_lite = classify_lite`
`(sequential_1_input=img_array)['outputs']:` Se pasa la imagen procesada (img_array) como entrada al modelo a través de la firma.

El nombre de la entrada ('sequential_1_input') debe coincidir con el nombre de la capa de entrada en el modelo Keras. Se obtiene la salida de la predicción ('outputs').

**Aplicar Softmax a las Predicciones:**

`score_lite = tf.nn.softmax(predictions_lite): `Aplica la
función de activación softmax para obtener las probabilidades de cada clase, lo que permite identificar la clase más probable.

** Imprimir la Predicción:**

Se muestra la clase predicha y la probabilidad asociada.

`class_names[np.argmax(score_lite)]:` Obtiene el nombre de la clase con la mayor probabilidad.

`100 * np.max(score_lite):` Muestra la probabilidad en porcentaje.

**Comparar Predicciones:**

`print(np.max(np.abs(predictions - predictions_lite))):`

 Se compara la predicción generada por el modelo original en TensorFlow con la predicción obtenida de TensorFlow Lite. Los resultados deberían ser casi idénticos, con una diferencia mínima debido a la conversión.

**Explicación de Firmas en TensorFlow Lite:**

Las firmas en TensorFlow Lite definen cómo interactuar con el modelo, especificando qué entradas y salidas se utilizan para la inferencia.

En este ejemplo, la firma predeterminada es serving_default, que es la utilizada por defecto cuando se convierte un modelo sin configuraciones adicionales. Al llamar a get_signature_runner('serving_default'), accedemos a la función predeterminada que puede recibir las entradas del modelo y devolver las salidas correspondientes.

In [ ]:
import tensorflow as tf
import numpy as np

# Ruta al modelo TensorFlow Lite guardado
TF_MODEL_FILE_PATH = 'model.tflite'

# Cargar el modelo TFLite usando el intérprete
interpreter = tf.lite.Interpreter(model_path=TF_MODEL_FILE_PATH)

# Obtener la lista de firmas del modelo convertido
signature_list = interpreter.get_signature_list()
print("Firmas del modelo:", signature_list)

# Usar la firma predeterminada (en este caso 'serving_default')
classify_lite = interpreter.get_signature_runner('serving_default')

# Realizar la inferencia: pasar la imagen tensorizada (img_array) al modelo TFLite
predictions_lite = classify_lite(sequential_1_input=img_array)['outputs']

# Aplicar la activación softmax para obtener probabilidades
score_lite = tf.nn.softmax(predictions_lite)

# Imprimir la clase predicha y la probabilidad de la predicción
print(
    "Esta imagen probablemente pertenece a {} con una probabilidad de {:.2f}%."
    .format(class_names[np.argmax(score_lite)], 100 * np.max(score_lite))
)

# Comparar las predicciones obtenidas por el modelo Keras original y el modelo TFLite
print(np.max(np.abs(predictions - predictions_lite)))  # Debería ser casi 0



Similar to what you did earlier in the tutorial, you can use the TensorFlow Lite model to classify images that weren't included in the training or validation sets.

You have already tensorized that image and saved it as `img_array`. Now, pass it to the first argument (the name of the `'inputs'`) of the loaded TensorFlow Lite model (`predictions_lite`), compute softmax activations, and then print the prediction for the class with the highest computed probability.

In [ ]:
predictions_lite = classify_lite(sequential_1_input=img_array)['outputs']
score_lite = tf.nn.softmax(predictions_lite)

In [ ]:
print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score_lite)], 100 * np.max(score_lite))
)

The prediction generated by the lite model should be almost identical to the predictions generated by the original model:

In [ ]:
print(np.max(np.abs(predictions - predictions_lite)))

Of the five classes—`'daisy'`, `'dandelion'`, `'roses'`, `'sunflowers'`, and `'tulips'`—the model should predict the image belongs to sunflowers, which is the same result as before the TensorFlow Lite conversion.
